# How a Neuron Learns: Gradient Descent and Backpropagation

Every neural network, from the small one in this notebook to the large models used for images and text, is built from one unit: a **neuron**. A neuron multiplies its inputs by weights, adds a bias and passes the result through an **activation function**. This notebook builds that idea up one step at a time, by hand with NumPy. The next notebook performs the same steps with Keras.

## Learning Objectives

By the end of this notebook, you should be able to:

- Compute the output of a single neuron from its inputs, weights, bias and activation function.
- Train a single neuron with gradient descent written by hand.
- Compare a linear neuron with a small ReLU network on non-linear data.
- Compute the gradients of a small ReLU network with backpropagation.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Fix the random seed so every run gives the same numbers
rng = np.random.default_rng(42)

# One plot style for the whole notebook
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
)

## 1. What a Neuron Computes

A neuron takes the $d$ input features of one example, $x_1$ to $x_d$, and does two things:

1. It computes a **weighted sum** $z$: each input is multiplied by its **weight**, and a **bias** $b$ is added.
2. It passes $z$ through an **activation function** $f$ to produce its output $\hat{y}$.

$$z = w_1 x_1 + w_2 x_2 + \dots + w_d x_d + b, \qquad \hat{y} = f(z)$$

The weights say how much each input matters, the bias shifts the result, and the activation decides what the neuron passes on. Training a network means finding good values for all the weights and biases.

The figure shows a neuron with three inputs:

<img src="assets/neuron.png" width="720">

Neural networks compute with **NumPy arrays**. An array created with [`np.array`](https://numpy.org/doc/stable/reference/generated/numpy.array.html) is a grid of numbers without labels, and its [`shape`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.shape.html) gives the number of rows and columns. The examples form an array `X` with one row per example and one column per input feature, as in a dataset; the capital $X$ marks a matrix. Arrays have no labels, so this notebook shows them as DataFrames.

In [ ]:
# Three examples: one row per example, one column per input feature
X = np.array(
    [
        [1.5, -0.5, 2.0],  # example 1
        [0.5, 0.5, 1.0],  # example 2
        [-1.0, 2.0, 0.5],  # example 3
    ]
)
print(f"X has shape {X.shape}: 3 examples (rows) and 3 input features (columns)")

In [ ]:
# The array has no labels; a DataFrame adds them so the table is easier to read
examples = ["example 1", "example 2", "example 3"]
features = ["x1", "x2", "x3"]
pd.DataFrame(X, index=examples, columns=features)

The neuron stores one weight per input feature, the same for every example, and one bias. The weights $w_1$ to $w_3$ form a **column** with one row per input feature: [`reshape(3, 1)`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html) arranges the three numbers into 3 rows and 1 column. A column is used because in section 3 each neuron of a layer becomes one column of the layer's weight matrix.

In [ ]:
# The neuron: one weight per input feature, plus one bias
weights = np.array([0.4, 1.2, -0.3]).reshape(3, 1)  # a column: one row per input feature
b = 0.1
print(f"weights has shape {weights.shape}: one row per input feature, one column for the neuron")
pd.DataFrame(weights, index=["w1", "w2", "w3"], columns=["neuron"])

The weighted sum of one example multiplies each feature by its weight and adds up the products: the **dot product** of the example's row with the weight column. [`np.dot`](https://numpy.org/doc/stable/reference/generated/numpy.dot.html) computes it for every row of `X` at once.

Every dot product follows one shape rule. In (3 × **3**) · (**3** × 1) the two **inner dimensions** must be equal, here the 3 input features, and the two **outer dimensions** give the shape of the result, (3 × 1): one weighted sum per example. Adding the bias gives $z$. In the figure each example has its own colour:

<img src="assets/numpy_dot_product_neuron.png" width="800">

In [ ]:
dot_product = np.dot(X, weights)  # (3 × 3) · (3 × 1) = (3 × 1)
z = dot_product + b  # one weighted sum per example
print(f"z has shape {z.shape}: one row per example, one column")

sums = pd.DataFrame(dot_product, index=examples, columns=["dot product"])
sums["z = dot product + b"] = z
sums.round(2)

Adding the bias of 0.10 turns the dot products -0.60, 0.50 and 1.85 into $z = -0.50$, $0.60$ and $1.95$.

### Activation Functions

Without an activation, a neuron only computes a weighted sum, which is a linear function of its inputs. Stacking such neurons gives another weighted sum, so the whole network would still draw only straight lines. The activation function adds the bend that lets a network learn curved relationships. Two common choices are:

- **ReLU** (rectified linear unit), $\max(0, z)$: it passes positive values unchanged and sets negative values to zero. It is the default for hidden layers, the layers between the input and the output (section 3).
- **Sigmoid**, $1 / (1 + e^{-z})$: it squeezes any number into the range 0 to 1, which is why it is used for the output of binary classifiers.

In [ ]:
def relu(z):
    """Return z where it is positive and 0 elsewhere (rectified linear unit)."""
    return np.maximum(0, z)


def sigmoid(z):
    """Squeeze z into the range 0 to 1 with the logistic function."""
    return 1 / (1 + np.exp(-z))


activation_table = pd.DataFrame(z, index=examples, columns=["z"])
activation_table["ReLU(z)"] = relu(z)
activation_table["sigmoid(z)"] = sigmoid(z)
activation_table.round(2)

For example 1 the weighted sum is negative ($z = -0.50$), so ReLU outputs 0 and the neuron passes nothing on for this example. Examples 2 and 3 have positive sums, which ReLU passes on unchanged. The sigmoid outputs 0.38 for example 1, below 0.5, and 0.65 and 0.88 for the other two. The plots below show each function over a range of inputs, with the three examples marked as numbered dots.

In [ ]:
z_range = np.linspace(-5, 5, 200)
z_values = z.flatten()  # the three weighted sums as a flat list, for plotting
activations = {
    "Identity: z": (z_range, z_values),
    "ReLU: max(0, z)": (relu(z_range), relu(z_values)),
    "Sigmoid: 1 / (1 + exp(-z))": (sigmoid(z_range), sigmoid(z_values)),
}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharex=True)
for ax, (name, (values, at_z)) in zip(axes, activations.items()):
    ax.plot(z_range, values, color="#2f6fb0", linewidth=2)
    ax.scatter(z_values, at_z, color="#c4423f", zorder=3, label="the three examples")
    for number, (z_i, y_i) in enumerate(zip(z_values, at_z), start=1):
        ax.annotate(str(number), (z_i, y_i), textcoords="offset points", xytext=(-4, 7))
    ax.set_title(name)
    ax.set_xlabel("weighted sum z")
axes[0].set_ylabel("output f(z)")
axes[0].legend()
fig.tight_layout()
plt.show()

### Try It: Compute a Neuron's Output

A fourth example has the features $x_1 = -1$, $x_2 = 0.5$ and $x_3 = 2$. Compute its weighted sum with the neuron's `weights` and `b`, and the output of the neuron after ReLU.

In [ ]:
x_new = np.array([[-1.0, 0.5, 2.0]])  # one example: 1 row, 3 input features

# TODO: the weighted sum of x_new, a (1 × 1) array
z_new = ...
# TODO: the output of the neuron after ReLU
output_new = ...

print(f"z = {z_new[0, 0]:.2f}, ReLU(z) = {output_new[0, 0]:.2f}")

## 2. Training One Neuron with Gradient Descent

A neuron with the identity activation computes $\hat{y} = w x + b$, which is exactly the linear regression model you already know. That makes it a good first model to train by hand.

The data below follows the line $y = 2x + 1$ plus some noise, so a well-trained neuron should end up with $w \approx 2$ and $b \approx 1$.

In [ ]:
line_data = pd.DataFrame({"x": rng.uniform(-3, 3, size=100)})
line_data["y"] = 2 * line_data["x"] + 1 + rng.normal(0, 1, size=100)
x_plot = np.linspace(-3, 3, 100)  # evenly spaced points for drawing lines

In [ ]:
plt.scatter(line_data["x"], line_data["y"], alpha=0.6)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Noisy data around y = 2x + 1")
plt.show()

Training repeats four steps, shown in the figure for a single weight $w$. The panel on the right gives the formula of each step. The loss is the **mean squared error** (MSE), a single number for how wrong the predictions are, and the size of each update is set by the **learning rate** $\eta$.

<img src="assets/gradient_descent.png" width="700">

Predicting with the current weight $w_0$ (1) and measuring the loss (2) give one point on the bowl-shaped loss curve. The gradient (3) is the slope of the tangent at that point, and the update (4) moves $w$ a small step against the gradient, downhill to $w_1$. Repeating the four steps from $w_1$, $w_2$ and so on brings $w$ towards the minimum. The steps get shorter near the minimum because the slope gets flatter there.

Training starts from a **random** weight and bias. For a single neuron any start would do, because its loss curve has only one minimum. In a network the start matters: if all neurons of a layer began with the same weights, they would compute the same output, receive the same gradients and stay identical, so the whole layer would act like a single neuron. **Random starting weights break this symmetry**, which is why every deep learning library starts from random values. `train_neuron` draws its start from its own random generator with a fixed `seed`, so every call starts from the same point and the learning rates below are compared fairly.

`train_neuron` uses the shapes of section 1. The inputs are a one-column array `X` and the targets a column `y`, both with one row per example. The weight `w` is a (1 × 1) matrix, one row for the input and one column for the neuron, and `b` holds the neuron's one bias. The prediction is the dot product of `X` and `w`, plus `b`: (n × 1) · (1 × 1) gives (n × 1), one prediction per example.

Step 3 needs one gradient per parameter. Both sum over the $n$ examples:

| Parameter | Gradient | Shape |
|---|---|---|
| weight $w$ | $\frac{\partial L}{\partial w} = \frac{2}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)\, x_i$ | (1 × n) · (n × 1) = (1 × 1), like `w` |
| bias $b$ | $\frac{\partial L}{\partial b} = \frac{2}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)$ | one value, like `b` |

The weight gradient multiplies the error of every example by its input and adds up the products, which is a dot product. For the inner dimensions to match, `X` is transposed first: [`X.T`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.T.html) is the **transpose** of `X`, with rows and columns swapped, so the code computes it as `2 * np.dot(X.T, error) / len(y)`. The bias gradient has no input to multiply by, so it is the mean of the errors times 2.

In [ ]:
def train_neuron(X, y, learning_rate, n_steps=50, seed=0):
    """Fit y_hat = X · w + b with gradient descent, starting from a random w and b.

    X has one row per example and one column per input feature, y one row per
    example. The seed fixes the random start, so every call starts from the same
    point. Returns the learned weights w (one row per input feature, one column),
    the learned bias b (one value) and the list of MSE losses, one per step.
    """
    start = np.random.default_rng(seed)  # its own generator, so the data cells stay unchanged
    start_values = start.normal(0, 1, size=2)  # the random starting weight and bias
    w = start_values[:1].reshape(1, 1)  # 1 input × 1 neuron, like the weights in section 1
    b = start_values[1:]  # one bias for the neuron
    losses = []
    for _ in range(n_steps):
        # TODO: 1. predict: the dot product of X and w, plus b
        y_hat = ...
        error = y_hat - y
        # TODO: 2. measure the loss: the mean of the squared errors (MSE)
        losses.append(...)
        grad_w = 2 * np.dot(X.T, error) / len(y)  # 3. compute the gradients: (1 × n) · (n × 1) = (1 × 1)
        # TODO: the gradient of b: 2 times the mean of the errors (use .mean(axis=0) to keep one value, like b)
        grad_b = ...
        # TODO: 4. update w and b: move each one step of learning_rate times its gradient, against the gradient
        w = ...
        b = ...
    return w, b, losses

In [ ]:
X_line = line_data[["x"]].to_numpy()  # one row per example, one column: shape (100, 1)
y_line = line_data[["y"]].to_numpy()  # the targets as a column too: shape (100, 1)

w_fit, b_fit, losses = train_neuron(X_line, y_line, learning_rate=0.1)
print(f"Learned w = {w_fit[0, 0]:.2f}, b = {b_fit[0]:.2f}")
print(f"Loss: first step {losses[0]:.2f}, last step {losses[-1]:.2f}")

In [ ]:
fig, (ax_loss, ax_fit) = plt.subplots(1, 2, figsize=(12, 4))

ax_loss.plot(losses)
ax_loss.set_xlabel("Step")
ax_loss.set_ylabel("MSE loss")
ax_loss.set_title("Loss during training")

ax_fit.scatter(line_data["x"], line_data["y"], alpha=0.6, label="Data")
ax_fit.plot(x_plot, w_fit[0, 0] * x_plot + b_fit[0], color="black", label="Trained neuron")
ax_fit.set_xlabel("x")
ax_fit.set_ylabel("y")
ax_fit.set_title("Fitted line")
ax_fit.legend()

fig.tight_layout()
plt.show()

Starting from the random values w = 0.13 and b = -0.13, the neuron learns w = 2.01 and b = 0.99 in 50 steps, very close to the line y = 2x + 1 that generated the data. The loss falls from 11.40 at the first step to 0.96 at the last. It cannot reach zero, because the data contains random noise with a variance of 1, which no line can predict.

### The Learning Rate

The learning rate is the most important setting of gradient descent. The cell below trains the same neuron with three learning rates and plots the loss on a logarithmic scale.

In [ ]:
for learning_rate in [0.01, 0.1, 0.5]:
    _, _, lr_losses = train_neuron(X_line, y_line, learning_rate=learning_rate)
    plt.plot(lr_losses, label=f"learning rate = {learning_rate}")
    print(f"learning rate {learning_rate}: final loss {lr_losses[-1]:.3g}")

plt.yscale("log")
plt.xlabel("Step")
plt.ylabel("MSE loss (log scale)")
plt.legend()
plt.show()

- With a learning rate of **0.01** the loss is still 1.20 after 50 steps. Each step is so small that training has not finished.
- With **0.1** the loss reaches 0.959, the lowest value a straight line can achieve on this data.
- With **0.5** every step overshoots the minimum by more than it corrects, so the loss grows to about $10^{23}$. Training has **diverged**.

### Try It: Tune the Learning Rate

Use `train_neuron` from section 2 on the straight-line data. Find a learning rate and a number of steps that reach a final loss below 1.0 in as few steps as possible.

In [ ]:
# TODO: choose a learning rate and a number of steps
my_learning_rate = ...
my_n_steps = ...

In [ ]:
w_try, b_try, losses_try = train_neuron(X_line, y_line, learning_rate=my_learning_rate, n_steps=my_n_steps)
print(f"w = {w_try[0, 0]:.2f}, b = {b_try[0]:.2f}, final loss = {losses_try[-1]:.3f}")

## 3. When One Neuron Is Not Enough

Now the data follows a curve, $y = x^2$ plus noise. A single linear neuron can only draw a straight line, regardless of the weights it learns.

In [ ]:
curve_data = pd.DataFrame({"x": rng.uniform(-3, 3, size=200)})
curve_data["y"] = curve_data["x"] ** 2 + rng.normal(0, 0.5, size=200)

In [ ]:
X_curve = curve_data[["x"]].to_numpy()  # one row per example, one column: shape (200, 1)
y_curve = curve_data[["y"]].to_numpy()  # the targets as a column: shape (200, 1)

w_c, b_c, curve_losses = train_neuron(X_curve, y_curve, learning_rate=0.1, n_steps=200)
print(f"Linear neuron: final MSE {curve_losses[-1]:.2f}")

In [ ]:
plt.scatter(curve_data["x"], curve_data["y"], alpha=0.5, label="Data")
plt.plot(x_plot, w_c[0, 0] * x_plot + b_c[0], color="black", label="Linear neuron")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

### Building a Curve from ReLUs

The linear neuron above draws a straight line, so it cannot follow the parabola. A ReLU neuron draws a line with **one bend**. Several ReLU neurons, each bending at a different point, add up to a curve made of straight pieces.

The neurons of a network are arranged in **layers**. The **input layer** holds the features, here the single number $x$; it does no computation. The **output layer** produces the prediction. Every layer in between is a **hidden layer**: its outputs are neither inputs nor targets in the data, so you never see them directly.

The network below approximates $y = x^2$ with one hidden layer of four ReLU neurons and hand-picked weights:

1. The input $x$ (one number, here $x = 2$) goes to all four hidden neurons.
2. Each hidden neuron computes $z = w \cdot x + b$ and then $h = \text{ReLU}(z)$.
3. The output neuron multiplies each $h$ by its output weight $v$ and adds the results: $\hat{y} = 4.5$, close to the true value $2^2 = 4$.

<img src="assets/relu_curve.png" width="800">

Information flows in one direction, from input to output, so this is a **feed-forward network**, and computing a prediction this way is a **forward pass**. The grey numbers trace $x = 2$; the cell below runs the same network for every $x$ from -3 to 3, with each neuron in its diagram colour.

In [ ]:
# The four hidden neurons: output weight v, input weight w, bias b and plot colour
relu_neurons = {
    "ReLU(x)": (1.5, 1, 0.0, "#2f6fb0"),
    "ReLU(-x)": (1.5, -1, 0.0, "#17a2b8"),
    "ReLU(x - 1.5)": (3, 1, -1.5, "#7b52ab"),
    "ReLU(-x - 1.5)": (3, -1, -1.5, "#8c564b"),
}
pieces = pd.DataFrame({name: v * relu(w * x_plot + b) for name, (v, w, b, _) in relu_neurons.items()})
relu_sum = pieces.sum(axis=1)  # the output neuron adds the four pieces

In [ ]:
# Left column: one plot per neuron, stacked vertically. Right column: their sum
fig = plt.figure(figsize=(12, 8))
grid = fig.add_gridspec(len(relu_neurons), 2)
ax_sum = fig.add_subplot(grid[:, 1])
for row, (name, (v, w, b, colour)) in enumerate(relu_neurons.items()):
    bend = -b / w + 0  # where w * x + b = 0; "+ 0" turns -0.0 into 0.0
    ax = fig.add_subplot(grid[row, 0], sharex=ax_sum)
    ax.plot(x_plot, pieces[name], color=colour, linewidth=2.2)
    ax.axvline(bend, color=colour, linestyle=":", linewidth=1.2)
    ax.set_title(f"{v} · {name}: bends at x = {bend:g}", color=colour, fontsize=10, loc="left")
    ax.set_ylim(-0.5, 9.5)
    if row < len(relu_neurons) - 1:
        ax.tick_params(labelbottom=False)
ax.set_xlabel("x")

for bend in [-1.5, 0, 1.5]:  # the bend points of the four neurons
    ax_sum.axvline(bend, color="grey", linestyle=":", linewidth=1)
exact_x = np.array([-3, -1.5, 0, 1.5, 3])  # where the sum meets the parabola exactly
ax_sum.scatter(curve_data["x"], curve_data["y"], color="grey", alpha=0.3, s=15, label="Data")
ax_sum.plot(x_plot, x_plot**2, color="#2e9467", linestyle="--", linewidth=1.8, label="True curve y = x²")
ax_sum.plot(x_plot, relu_sum, color="black", linewidth=2.4, label="Sum of the four neurons")
ax_sum.scatter(exact_x, exact_x**2, color="#c4423f", s=45, zorder=3, label="Exact at x = 0, ±1.5, ±3")
ax_sum.set_title("Their sum approximates y = x²")
ax_sum.set_xlabel("x")
ax_sum.legend()

fig.tight_layout()
plt.show()

The red dots mark the five points where the sum equals the parabola exactly ($x = 0$, $\pm 1.5$, $\pm 3$). Between them the straight pieces sit slightly above the parabola: that gap is the approximation error.

This is what a hidden layer does: each neuron contributes one bent line, and the output neuron adds them up. More neurons allow more bends. A real network *learns* where to put the bends and how steep each piece is; section 4 shows how.

### How Many Weights Does a Hidden Layer Need?

Every neuron in a hidden layer receives **all** the inputs of the layer and has its own weight for each one. The weights of the layer therefore form a matrix $W$:

- **Rows: one per input** coming into the layer. For the first hidden layer this is the number of features $d$; for any later layer it is the number of neurons in the layer before, because their outputs are its inputs.
- **Columns: one per neuron** in the layer. The number of neurons is a hyperparameter that you choose, as the argument `units` of a Keras `Dense` layer.
- **Biases: one per neuron**, so a layer has as many biases as $W$ has columns.

The number of training examples does not appear anywhere: the same weights are used for every example.

As an example, send the three examples $X$ from section 1 into a hidden layer of 4 neurons. $W$ has 3 rows and 4 columns, so the layer has $3 \times 4 = 12$ weights plus 4 biases. The weight $w_{kj}$ connects input $k$ to hidden neuron $j$. In the figure, each neuron's connections and its column of $W$ share one colour, and the highlighted line from $x_2$ to neuron 3 is the cell $w_{23}$:

<img src="assets/hidden_layer_weights.png" width="800">

Column $j$ holds all the weights of hidden neuron $j$, and the neuron computes the same weighted sum as the single neuron in section 1:

$$z_j = x_1 w_{1j} + x_2 w_{2j} + x_3 w_{3j} + b_j$$

The cell below stores $W$ as an array. Neuron 1 reuses the weight column from section 1; the other values are hand-picked. `np.dot(X, W)` computes one dot product for every pair of a row of `X` and a column of `W`: all four weighted sums for all three examples at once. The shape rule from section 1 applies: (3 × 3) · (3 × 4) gives (3 × 4). If the inner dimensions differ, as in `np.dot(W, X)` with (3 × 4) · (3 × 3), NumPy stops with a `ValueError`. In the figure each neuron has its own colour:

<img src="assets/numpy_dot_product.png" width="800">

In [ ]:
# Hidden layer of 4 neurons: one row per input feature, one column per neuron
W = np.array(
    [
        [0.4, -0.5, 0.3, 1.0],  # weights of input x1
        [1.2, 0.2, -0.4, 0.5],  # weights of input x2
        [-0.3, 0.1, 0.6, -0.2],  # weights of input x3
    ]
)
hidden_biases = np.array([0.1, 0.0, -0.2, 0.3])  # one bias per neuron

neurons = ["neuron 1", "neuron 2", "neuron 3", "neuron 4"]
print(f"X {X.shape}: one row per example, one column per input")
print(pd.DataFrame(X, index=examples, columns=features), end="\n\n")
print(f"W {W.shape}: one row per input, one column per neuron")
pd.DataFrame(W, index=features, columns=neurons)

In [ ]:
z_hidden = np.dot(X, W) + hidden_biases  # all weighted sums at once
h_hidden = relu(z_hidden)  # the outputs of the hidden layer

print(f"z = np.dot(X, W) + b {z_hidden.shape}: one row per example, one column per neuron")
print(pd.DataFrame(z_hidden, index=examples, columns=neurons).round(2), end="\n\n")
print(f"ReLU(z) {h_hidden.shape}: the outputs of the hidden layer")
pd.DataFrame(h_hidden, index=examples, columns=neurons).round(2)

Neuron 1 has the weights and bias of the neuron in section 1, so its column reproduces $z = -0.50$, $0.60$ and $1.95$. Adding `hidden_biases`, an array of four biases, to the (3 × 4) result is called [**broadcasting**](https://numpy.org/doc/stable/user/basics.broadcasting.html): NumPy applies the shorter array to every row, so the bias of each neuron is added to its whole column. ReLU then sets every negative weighted sum to zero. With 100 examples, $X$ would have 100 rows and the output 100 rows, but $W$ would still be $3 \times 4$: the number of examples changes the shape of the data, never the shape of the weights.

The same rule applies layer by layer. If a second hidden layer of 2 neurons followed, it would receive the 4 outputs above, so its $W$ would be $4 \times 2$ with 2 biases. In general:

$$\text{parameters of a layer} = (\text{inputs}) \times (\text{neurons}) + (\text{neurons})$$

The ReLU network of this section has a single input $x$ and 4 hidden neurons, so its hidden $W$ is $1 \times 4$: the four weights $w_1$ to $w_4$, one per neuron. The output neuron receives the 4 hidden outputs, so its weights $v_1$ to $v_4$ form a $4 \times 1$ matrix. With the biases $b_1$ to $b_4$ and $c$, the network has $(1 \times 4 + 4) + (4 \times 1 + 1) = 13$ parameters.

### Try It: Count the Parameters

Notebook 03 builds a network for house prices with 8 input features, three hidden layers of 50 neurons each and one output neuron. Every neuron of a layer is connected to every neuron of the layer before, as the figure shows. Use the rule above to work out the shape of $W$ in the first hidden layer and the number of parameters of every layer. You can check your total there again: it is the number of trainable parameters that `model.summary()` prints in notebook 03.

<img src="assets/count_parameters_network.png" width="800">

In [ ]:
# TODO: the shape of W in the first hidden layer, as (rows, columns)
W_first_shape = ...
# TODO: the parameters (weights plus biases) of the four layers, from the first hidden layer to the output
params_per_layer = [..., ..., ..., ...]

print(f"Parameters per layer: {params_per_layer}, total {sum(params_per_layer):,}")

## 4. Backpropagation: How a Network Finds Its Gradients

Gradient descent needs the gradient of the loss with respect to every weight and bias. For the single neuron in section 2 those gradients could be written down directly. In a network, a hidden weight affects the loss only indirectly: it changes the output of its hidden neuron, which changes the prediction, which changes the loss. **Backpropagation** computes these indirect gradients with the **chain rule**, starting at the loss and working backwards through the layers. This is called the **backward pass**, and it is step 3 of the training loop from section 2.

<img src="assets/training_cycle.png" width="560">

The figure shows the four steps of training on the ReLU network from section 3. The forward pass runs from $x$ to the prediction $\hat{y}$; the backward pass runs the other way, from the loss back to the input weights and biases $w, b$ and the output weights and bias $v, c$. Steps 1, 2 and 4 already appear in sections 2 and 3; this section is about step 3.

### The Chain Rule

If $f$ depends on $u$ and $u$ depends on $x$, the effect of $x$ on $f$ is the product of the two local effects:

$$\frac{df}{dx} = \frac{df}{du} \cdot \frac{du}{dx}$$

For example, $f = (3x + 2)^2$ is $f = u^2$ with $u = 3x + 2$. Then $df/du = 2u$ and $du/dx = 3$, so $df/dx = 2(3x + 2) \cdot 3 = 6(3x + 2)$, which is 30 at $x = 1$. A network is a longer chain of the same kind: each layer is a function of the layer before it.

### The Chain Rule in the ReLU Network

Take the network from the diagram above. Hidden neuron $j$ (from 1 to 4) has an input weight $w_j$, a bias $b_j$ and an output weight $v_j$, and the output neuron has a bias $c$. The index $i$ counts the $n$ training examples, so $x_i$ is the input of example $i$ and $h_{ij}$ is the output of neuron $j$ for that example. The forward pass is a chain of four steps:

$$z_{ij} = w_j x_i + b_j, \qquad h_{ij} = \text{ReLU}(z_{ij}), \qquad \hat{y}_i = \sum_{j=1}^{4} v_j h_{ij} + c, \qquad L = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

The backward pass walks the same chain in reverse and multiplies one local derivative per step:

| Step backwards | Local derivative |
|---|---|
| ${\color{#2e9467}{\text{I. Loss to prediction}}}$ | ${\color{#2e9467}{\partial L / \partial \hat{y}_i = \frac{2}{n}(\hat{y}_i - y_i)}}$ |
| ${\color{#c47612}{\text{II. Prediction to output weight and output bias}}}$ | ${\color{#c47612}{\partial \hat{y}_i / \partial v_j = h_{ij}, \quad \partial \hat{y}_i / \partial c = 1}}$ |
| ${\color{#c47612}{\text{III. Prediction to hidden output}}}$ | ${\color{#c47612}{\partial \hat{y}_i / \partial h_{ij} = v_j}}$ |
| ${\color{#2f6fb0}{\text{IV. Hidden output to weighted sum (ReLU)}}}$ | ${\color{#2f6fb0}{\partial h_{ij} / \partial z_{ij} = 1 \text{ if } z_{ij} > 0 \text{, otherwise } 0}}$ |
| ${\color{#c2407e}{\text{V. Weighted sum to input weight and bias}}}$ | ${\color{#c2407e}{\partial z_{ij} / \partial w_j = x_i, \quad \partial z_{ij} / \partial b_j = 1}}$ |

The numerals and colours match the badges in the training-cycle figure. Multiplying the local derivatives along the chain and summing over the examples gives the gradient of each parameter:

| Parameter | Steps of the chain | Gradient | Code in `backward` |
|---|---|---|---|
| output weight $v_j$ | I, II | $\sum_{i=1}^{n} {\color{#2e9467}{\tfrac{2}{n}(\hat{y}_i - y_i)}} \cdot {\color{#c47612}{h_{ij}}}$ | `np.dot(h.T, d_y_hat)` |
| output bias $c$ | I, II | $\sum_{i=1}^{n} {\color{#2e9467}{\tfrac{2}{n}(\hat{y}_i - y_i)}} \cdot {\color{#c47612}{1}}$ | `d_y_hat.sum()` |
| input weight $w_j$ | I, III, IV, V | $\sum_{i=1}^{n} {\color{#2e9467}{\tfrac{2}{n}(\hat{y}_i - y_i)}} \cdot {\color{#c47612}{v_j}} \cdot {\color{#2f6fb0}{\text{ReLU}'(z_{ij})}} \cdot {\color{#c2407e}{x_i}}$ | `np.dot(X.T, d_z)` |
| bias $b_j$ | I, III, IV, V | $\sum_{i=1}^{n} {\color{#2e9467}{\tfrac{2}{n}(\hat{y}_i - y_i)}} \cdot {\color{#c47612}{v_j}} \cdot {\color{#2f6fb0}{\text{ReLU}'(z_{ij})}} \cdot {\color{#c2407e}{1}}$ | `d_z.sum(axis=0)` |

The output weight and the output bias are only two steps away from the loss; the input weights and biases sit further back. Each bias gradient is the matching weight gradient with the last factor replaced by 1, because a bias is added directly rather than multiplied by an input.

The ReLU derivative is a step: 1 where the neuron is active and 0 where it outputs 0. An inactive neuron passes no gradient back, so its input weight and bias get no update from that example.

The cells below write the forward and backward passes with NumPy, starting from the hand-picked values above. The input is the one-column `X_curve` from the linear neuron above, taken from the DataFrame with double brackets (`curve_data[["x"]]`), as Keras expects it. The input weights form the $1 \times 4$ matrix `W`, so `np.dot(X, W) + b` computes the hidden layer as in section 3: (200 × 1) · (1 × 4) gives (200 × 4), one row per example and one column per neuron. The biases `b` hold one value per neuron, the output weights `v` form a (4 × 1) column and `c` is a single number, so the predictions are a column like the targets `y_curve`. In `backward` every gradient comes out with the same shape as its parameter. Each sum over the examples is a dot product with a transposed array, as in `train_neuron`: `np.dot(X.T, d_z)` is (1 × 200) · (200 × 4) = (1 × 4), the shape of `W`. The comments name the table row each line uses.

In [ ]:
def forward(X, W, b, v, c):
    """Run a forward pass of the one-hidden-layer ReLU network for every example in X.

    X has one row per example and one column, the input x. W holds the input weights
    (1 input × 4 neurons), b one bias per hidden neuron, v the output weights
    (4 neurons × 1 output) and c is the output bias. Returns the weighted sums z and
    the hidden outputs h (one row per example, one column per neuron) and the
    predictions y_hat (one row per example).
    """
    z = np.dot(X, W) + b  # the hidden layer, as in section 3: z_ij = x_i * w_j + b_j
    h = relu(z)
    y_hat = np.dot(h, v) + c  # (n × 4) · (4 × 1) gives one prediction per example
    return z, h, y_hat

In [ ]:
def backward(X, y, z, h, y_hat, v):
    """Backpropagate the MSE loss through the network with the chain rule.

    Returns the gradients of W, b, v and c, each with the same shape as its parameter.
    """
    d_y_hat = 2 * (y_hat - y) / len(y)  # row I: dL/dy_hat, one row per example
    d_h = np.dot(d_y_hat, v.T)  # row III: times v_j, (n × 1) · (1 × 4) gives one column per neuron
    d_z = d_h * (z > 0)  # row IV: times the ReLU derivative, 1 if active and 0 otherwise
    grad_W = np.dot(X.T, d_z)  # row V: times x_i, summed over the examples: (1 × n) · (n × 4)
    grad_b = d_z.sum(axis=0)  # row V: times 1, summed over the examples
    grad_v = np.dot(h.T, d_y_hat)  # row II: h_ij times dL/dy_hat, summed: (4 × n) · (n × 1)
    grad_c = d_y_hat.sum()  # row II for c: dL/dy_hat times 1
    return grad_W, grad_b, grad_v, grad_c

In [ ]:
def mse(y_hat, y):
    """Return the mean squared error between predictions and targets."""
    return ((y_hat - y) ** 2).mean()

In [ ]:
# The hand-picked network from the diagram
W = np.array([[1.0, -1.0, 1.0, -1.0]])  # input weights: 1 input × 4 neurons
b = np.array([0.0, 0.0, -1.5, -1.5])  # biases: one per neuron
v = np.array([[1.5], [1.5], [3.0], [3.0]])  # output weights: 4 neurons × 1 output
c = 0.0  # output bias

z, h, y_hat = forward(X_curve, W, b, v, c)
grad_W, grad_b, grad_v, grad_c = backward(X_curve, y_curve, z, h, y_hat, v)
print(f"MSE of the hand-picked network: {mse(y_hat, y_curve):.3f}")
print(f"Gradient of the output bias c: {grad_c:.3f}")

# Row 0 of grad_W and column 0 of grad_v hold one value per neuron
pd.DataFrame({"w": grad_W[0], "b": grad_b, "v": grad_v[:, 0]}, index=neurons).round(3)

The hand-picked network has an MSE of 0.436. Every gradient in the output above is positive, and so is the gradient of the output bias (0.862). A positive gradient means that increasing the parameter would increase the loss, so gradient descent will decrease it. This matches the plot above: the straight pieces sit above the parabola, so the predictions are too high on average.

### Training the Network with Backpropagation

Putting the forward pass, the backward pass and the update together gives the same four-step loop as in section 2, now for every parameter of the network. Write the loop in the cell below: it starts from the hand-picked weights and runs 500 steps of gradient descent. `forward` returns `z`, `h` and `y_hat`, `backward` returns the gradients of `W`, `b`, `v` and `c` in that order, and each update is the same as in `train_neuron`.

In [ ]:
learning_rate = 0.05
W_trained, b_trained, v_trained, c_trained = W.copy(), b.copy(), v.copy(), c
bp_losses = []
# TODO: repeat the four training steps 500 times
for _ in range(...):
    # TODO: 1. forward pass: call forward with X_curve and the current trained parameters
    z, h, y_hat = ...
    # TODO: 2. measure the loss with mse and append it to bp_losses
    ...
    # TODO: 3. backward pass: call backward to get the four gradients
    grad_W, grad_b, grad_v, grad_c = ...
    # TODO: 4. update every parameter: subtract learning_rate times its gradient
    W_trained = ...
    b_trained = ...
    v_trained = ...
    c_trained = ...

In [ ]:

print(f"MSE: first step {bp_losses[0]:.3f}, last step {bp_losses[-1]:.3f}")
print("Bend points (x where w * x + b = 0):")

In [ ]:
X_plot = x_plot.reshape(-1, 1)  # the plotting points as a column, like X_curve (-1: as many rows as needed)
plt.scatter(curve_data["x"], curve_data["y"], alpha=0.3, label="Data")
plt.plot(x_plot, forward(X_plot, W, b, v, c)[2], color="grey", linestyle="--", label="Hand-picked weights")
plt.plot(
    x_plot,
    forward(X_plot, W_trained, b_trained, v_trained, c_trained)[2],
    color="black",
    label="After backpropagation",
)
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

After 500 steps the MSE falls from 0.436 to 0.227, about the variance of the noise added to the data (0.25). The bend points have moved from 0, 0, 1.5 and -1.5 to 0.38, -0.22, 1.64 and -1.30: the network has learnt where to bend as well as how steeply. Backpropagation adjusted all 13 parameters at once (three per hidden neuron plus the output bias).

Deep learning libraries such as Keras run exactly this loop. They do not need a hand-written `backward` function, because they apply the chain rule automatically to every operation in the forward pass (**automatic differentiation**). The next notebook trains the same kind of network with Keras. The References at the end list complete implementations from scratch for further reading.

## Summary

- A neuron computes a weighted sum of its inputs plus a bias, then applies an activation function.
- With no activation, a single neuron is linear regression, and gradient descent finds its weights by repeating predict, measure the loss, compute the gradients, update.
- The learning rate sets the step size: too small and training is slow, too large and the loss grows instead of shrinking.
- A feed-forward network computes a prediction with a forward pass: the outputs of each layer become the inputs of the next, from the input layer to the output layer.
- A hidden layer of ReLU neurons adds up bent lines, which lets a network fit curved relationships that one linear neuron cannot.
- The weights of a layer form a matrix with one row per input and one column per neuron, plus one bias per neuron. The number of examples does not change it.
- Backpropagation computes the gradient of every weight with the chain rule, starting at the loss and working backwards through the layers. An inactive ReLU neuron passes no gradient back.
- Libraries such as Keras run the same training loop for you and compute the gradients with automatic differentiation.

Next, continue with [02 - The Keras Workflow: Configure, Compile, Fit, Evaluate, Predict](02_keras_workflow.ipynb).

## References

- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd edition, chapter 10.
- Chollet, F. and Watson, M. (2025). *Deep Learning with Python*, 3rd edition, chapter 2: The mathematical building blocks of neural networks. [Free to read online](https://deeplearningwithpython.io/).
- Howard, J. [How does a neural net really work?](https://www.kaggle.com/code/jhoward/how-does-a-neural-net-really-work) Kaggle notebook from the fast.ai course.
- 3Blue1Brown. [Neural networks video series](https://www.3blue1brown.com/topics/neural-networks): visual explanations of neurons, gradient descent and backpropagation.
- Nielsen, M. (2015). [How the backpropagation algorithm works](http://neuralnetworksanddeeplearning.com/chap2.html), chapter 2 of *Neural Networks and Deep Learning*.
- Stanford CS231n. [Putting it together: Minimal Neural Network Case Study](https://cs231n.github.io/neural-networks-case-study/): a complete network written in NumPy, with matrices, a softmax output and the log loss.
- Zhou, V. (2019). [Machine Learning for Beginners: An Introduction to Neural Networks](https://victorzhou.com/blog/intro-to-neural-networks/): a neuron and a small network built and trained from scratch in Python.
- Karpathy, A. [The spelled-out intro to neural networks and backpropagation: building micrograd](https://karpathy.ai/zero-to-hero.html).